# Learn `symproof` in practice

## A hidden-assumption hunt in Stellar's Neural Quorum Governance

This notebook is a storyboard companion to [`examples/nqg_voting.py`](nqg_voting.py). It applies `symproof` to a real governance system — Stellar's [Neural Quorum Governance (NQG)](https://github.com/stellar/stellar-community-fund-contracts) — and watches the proof framework surface assumptions latent in the code.

It mirrors the structure of [`examples/conviction_voting.ipynb`](conviction_voting.ipynb) deliberately. If you've worked through that notebook, the rhythm here will feel familiar; if you haven't, you can read either one first.

**What you'll learn**

1. The four-step rhythm of every `symproof` script: declare symbols → frame axioms → bind hypothesis → build script and seal.
2. How `seal()` refuses to certify a proof whose dependencies aren't declared — and why that refusal is the framework's most useful feature.
3. Two concrete failure-and-fix moments, surfaced verbatim, that show how the system pushes back when the math is incomplete.
4. A worked example whose punchline — the PageRank damping constant `d < 1` — is a load-bearing constraint that buys ergodicity (well-defined stationary distribution) but is rarely treated as a proof obligation in practice.

**Audience.** You've read the symproof README and want to feel how the framework behaves on an unfamiliar problem. You don't need a math background — every claim is explained in plain English first.


## Acknowledgment

This walkthrough is a deliberate clone of the conviction-voting walkthrough by [BlockScience](https://blockscience.com/) and [`heph789`](https://github.com/heph789), adapted to a different governance design. The conviction-voting notebook taught us the four-step rhythm; this one re-runs the rhythm on Neural Quorum Governance to make the pattern transferable.

The target system — Stellar's Neural Quorum Governance — was designed by the [Stellar Development Foundation](https://stellar.org) for the [Stellar Community Fund](https://stellarcommunityfund.gitbook.io/scf-handbook/community-involvement/governance/neural-quorum-governance). All bugs and mis-statements about the system are this notebook's fault, not theirs.


## Setup

```bash
uv sync --group notebooks
uv run jupyter lab examples/nqg_voting.ipynb
```

If you don't have `uv`, a plain venv works too:

```bash
python3 -m venv .venv
.venv/bin/pip install -e . ipykernel
.venv/bin/python -m ipykernel install --user --name symproof --display-name symproof
```

All cells execute deterministically. The `bundle_hash` values printed below should reproduce bit-for-bit; if you get different hashes, your `sympy` version probably differs.


In [1]:
import sympy
from sympy import Rational, Symbol

import symproof
from symproof import (
    Axiom,
    AxiomSet,
    LemmaKind,
    ProofBuilder,
    seal,
    unevaluated,
)

print(f"symproof version: {symproof.__version__}")
print(f"sympy version:    {sympy.__version__}")


symproof version: 0.1.0
sympy version:    1.14.0


## The four-step rhythm

Every `symproof` script follows the same four steps. Burn the rhythm into muscle memory and the framework gets out of your way.

1. **Declare symbols** with assumptions matching their domain (`Symbol("x", positive=True)`).
2. **Frame axioms** — the things you're accepting as true about the system, hashed together into an `AxiomSet`.
3. **Bind a hypothesis** to the axioms — what you intend to prove, carrying the axiom set's hash.
4. **Build a `ProofScript`** of lemmas and `seal()` it. The seal step re-verifies every lemma and produces a deterministic `bundle_hash`.

Before we touch NQG, the DeFi library has a one-line worked example: [`fee_complement_positive`](../symproof/library/defi.py) proves `1 - fee > 0` from `0 < fee < 1`. We run it as a warm-up to confirm `symproof` is wired up, and to see what a sealed bundle looks like.


In [2]:
from symproof.library.defi import fee_complement_positive

with unevaluated():
    fee = Symbol("fee", positive=True)
    pool_axioms = AxiomSet(name="amm_pool", axioms=(
        Axiom(name="fee_pos",   expr=fee > 0),
        Axiom(name="fee_lt_1",  expr=fee < 1),
    ))

fee_bundle = fee_complement_positive(pool_axioms, fee)
print(f"Claim:        1 - fee > 0")
print(f"Bundle hash:  {fee_bundle.bundle_hash}")
print(f"Status:       {fee_bundle.proof_result.status.value}")


Claim:        1 - fee > 0
Bundle hash:  068603dca03c09932b783ccb3bb704b9ddb2427f8e6a0ca6e64f1b6fa4050b0f
Status:       VERIFIED


## Switching domains: Neural Quorum Governance

[Neural Quorum Governance](https://stellarcommunityfund.gitbook.io/scf-handbook/community-involvement/governance/neural-quorum-governance) is the Stellar Community Fund's voting framework. Each voter's voting power isn't just "one wallet, one vote" — it's computed by composing several **neurons**, each contributing a score based on a different signal:

```mermaid
flowchart LR
    subgraph Inputs
      A[Discord roles]
      B[Past voting<br/>history]
      C[Trust graph<br/>of who-trusts-whom]
      D[Retro outcomes<br/>of past votes]
    end

    A --> N1[AssignedReputation<br/>neuron]
    B --> N2[PriorVotingHistory<br/>neuron]
    C --> N3[TrustGraph<br/>neuron]
    C --> N4[TrustHistory<br/>neuron]
    D --> N5[RetroVoteQuality<br/>neuron]

    N1 --> AGG[On-chain<br/>aggregator<br/>Sum / Product]
    N2 --> AGG
    N3 --> AGG
    N4 --> AGG
    N5 --> AGG

    AGG --> VP[Final voting power<br/>per voter]
    VP --> SCF[SCF Token<br/>balance]
    VP --> GOV[Soroban Governor<br/>proposal vote]
```

The off-chain Rust crate (`neurons/`) computes each neuron's per-voter score, hands the JSON output to the on-chain `governance` contract, which applies layer aggregators (`Sum`, `Product`) using `I256` fixed-point math.

Three of the neurons reduce to mathematical claims we can validate symbolically:

- **PriorVotingHistory** uses a generalised logistic decay and an `active-votes ratio` clipped from below at `0.5`.
- **TrustGraph** runs PageRank on the who-trusts-whom graph, normalises the result, and adds a "highly-trusted bonus" for voters trusted by top-decile peers.
- **TrustHistory** takes the *delta* between the current and previous round's trust score, mapped through a logistic penalty.

The exercise. We pick three small but load-bearing claims from the neurons code, frame them as hypotheses, and let `symproof` either certify them or — much more usefully — refuse, telling us which assumption we forgot to declare.


In [3]:
# NQG quantities, named to match the Rust source where possible.
# The constructor assumptions (positive=True, nonnegative=True) carry
# domain knowledge that we'll mirror as axioms in the next cell.
ratio       = Symbol("ratio",     nonnegative=True)   # active_votes / total_votes
score       = Symbol("score",     nonnegative=True)   # PageRank score in [0, 1]
bonus_pct   = Symbol("bonus_pct", positive=True)      # HIGHLY_TRUSTED_PERCENT_BONUS
d           = Symbol("d",         positive=True)      # PageRank damping factor
N           = Symbol("N",         positive=True, integer=True)  # node count

# The hard-coded floor from prior_voting_history.rs:
#     ACTIVE_VOTES_MIN_RATIO: f64 = 0.5
MIN_RATIO   = Rational(1, 2)


## Framing the axioms

Six axioms, each a fact about the system that the proofs below will rely on. The `inherited` column tells you whether the axiom comes from a stated design choice or from a deeper theorem we are not yet proving (those become foundation work later).

| # | Axiom | Why it's needed | Source in the Rust |
|---|---|---|---|
| N1 | `ratio ≥ 0` | vote-count ratio non-negative | `prior_voting_history.rs::calculate_active_votes_ratio` |
| N2 | `score ≥ 0` | PageRank scores live in `[0, 1]` after min-max normalisation | `trust_graph.rs::min_max_normalize_result` |
| N3 | `bonus_pct > 0` | the highly-trusted bonus is a positive constant (`15.0`) | `trust_graph.rs::HIGHLY_TRUSTED_PERCENT_BONUS` |
| N4 | `d > 0` | PageRank damping strictly positive | `trust_graph.rs::calculate_page_rank` (uses `0.85`) |
| **N5** | **`d < 1`** | **load-bearing for ergodicity — the punchline of this notebook** | `trust_graph.rs::calculate_page_rank` |
| N6 | `N > 0` | at least one node in the trust graph | `trust_graph.rs` (graph construction) |

**N5 is the one to watch.** The Rust code passes `0.85` and never checks the bound; mathematicians know `d < 1` is what makes PageRank's stationary distribution well-defined (Perron–Frobenius). We will prove one consequence of N5 — the teleport `(1−d)/N > 0` — and call out the other two consequences (irreducibility and uniqueness) as future foundation work.


In [4]:
with unevaluated():
    nqg_axioms = AxiomSet(
        name="nqg_neurons",
        axioms=(
            Axiom(name="ratio_nonneg", expr=ratio >= 0,
                  description="N1: vote-count ratio is non-negative"),
            Axiom(name="score_nonneg", expr=score >= 0,
                  description="N2: PageRank score is non-negative "
                              "(min-max normalisation forces it into [0,1])"),
            Axiom(name="bonus_pct_pos", expr=bonus_pct > 0,
                  description="N3: HIGHLY_TRUSTED_PERCENT_BONUS > 0 "
                              "(constant 15.0 in trust_graph.rs)"),
            Axiom(name="damping_pos", expr=d > 0,
                  description="N4: PageRank damping factor strictly positive"),
            Axiom(name="damping_lt_1", expr=d < 1,
                  description="N5: PageRank damping factor strictly below 1 "
                              "(load-bearing for ergodicity)"),
            Axiom(name="N_pos", expr=N > 0,
                  description="N6: at least one node in the trust graph"),
        ),
    )

print(f"Axiom set hash: {nqg_axioms.axiom_set_hash}")
print(f"Axioms:         {len(nqg_axioms.axioms)}")


Axiom set hash: d1e5e56ad03d08ffdf04a318da3a4605aac5515e8b973420b99a1cf1c5f15a40
Axioms:         6


## Hypothesis catalog

Per the symproof convention (see [CLAUDE.md](../CLAUDE.md)), each property gets its own hypothesis and its own sealed bundle. Independence is the feature: any one proof can be re-verified without the others. The hashes line up against requirement IDs in a traceability matrix.

Three proven in this notebook, three more framed as future foundation work:

| Req | Hypothesis | What we prove (in words) | Status |
|---|---|---|---|
| **REQ-RATIO-FLOOR** | `Max(0.5, ratio) ≥ 0.5` | `(active/total).max(0.5)` never returns less than 0.5 | proven |
| **REQ-TRUST-MONO** | `score' ≥ score` | the highly-trusted bonus never *decreases* anyone's score | proven |
| **REQ-TELEPORT-POS** | `(1−d)/N > 0` | every PageRank node gets a positive baseline rank | proven (uses N5) |
| REQ-PR-IRREDUCIBLE | the random walk reaches every node | needs Perron–Frobenius foundation | framed |
| REQ-PR-UNIQUE | the stationary distribution is unique | needs Perron–Frobenius foundation | framed |
| REQ-PR-CONVERGES | power iteration converges to it | needs Perron–Frobenius foundation | framed |

The "framed" rows show where this audit hits the boundary of what `symproof` can do today. Each one would become a foundation bundle that imports a Perron–Frobenius proof; the downstream PageRank proofs would then `seal(foundations=...)` against it, and any of the foundation's hidden assumptions become inherited axioms here.


In [5]:
score_new = score + (score / 100) * bonus_pct
teleport  = (1 - d) / N

h_mono = nqg_axioms.hypothesis(
    "trust_bonus_monotone",
    expr=sympy.Ge(score_new, score),
    description="score' = score*(1 + bonus_pct/100) >= score "
                "(highly-trusted bonus never decreases anyone's score)",
)
h_teleport = nqg_axioms.hypothesis(
    "teleport_positive",
    expr=teleport > 0,
    description="PageRank teleport (1-d)/N > 0",
)

# REQ-RATIO-FLOOR is built and bound by the library function in the next cell.
for h in (h_mono, h_teleport):
    print(f"  {h.name:<24s}  axioms={h.axiom_set_hash[:16]}...")


  trust_bonus_monotone      axioms=d1e5e56ad03d08ff...
  teleport_positive         axioms=d1e5e56ad03d08ff...


## Proof 1 — REQ-RATIO-FLOOR (the warm-up)

Target line in the Rust:

```rust
// neurons/src/prior_voting_history.rs
(active_votes_count / total_votes_count).max(ACTIVE_VOTES_MIN_RATIO)
//                                       ^ floors the ratio at 0.5
```

The `.max(0.5)` call returns the larger of its argument and `0.5`. We want to certify that the result is always `≥ 0.5`, no matter what the raw ratio comes out to.

### Why we care

The prior-voting-history neuron multiplies a round's weight by this clipped ratio. If the floor weren't enforced, a user with no active votes in some past round would get a `0` round-weight, collapsing their bonus to zero. The `0.5` floor is a designed-in baseline.

### Plain-English math

For any `x ≥ 0`, `Max(0.5, x) ≥ 0.5`. This is a tautology of `Max` — the result is always at least the larger of its arguments, and the larger of `0.5` and anything is at least `0.5`.

The library function `max_ge_first(ax, a, b)` proves `Max(a, b) ≥ a` in one call. We pass `(nqg_axioms, MIN_RATIO, ratio)` and out comes the sealed bundle.


In [6]:
from symproof.library.core import max_ge_first

ratio_bundle = max_ge_first(nqg_axioms, MIN_RATIO, ratio)

print(f"REQ-RATIO-FLOOR")
print(f"  hash:   {ratio_bundle.bundle_hash}")
print(f"  status: {ratio_bundle.proof_result.status.value}")


REQ-RATIO-FLOOR
  hash:   6143139b484d35431f96b732c0a777e68a2ca8b0e3141a4d07f2b69e3ec6d57a
  status: VERIFIED


## Failure-and-fix #1 — the load-bearing check fires

Now REQ-TRUST-MONO: prove that the trust-graph "highly-trusted bonus" never decreases anyone's score.

Target line in the Rust:

```rust
// neurons/src/trust_graph.rs::handle_highly_trusted_bonus
*res += (*res / 100.0) * percent_bonus;
```

That's the same as writing `score' = score · (1 + bonus_pct / 100)`. Plain English: if your score gets bumped up by some positive percent, your new score is at least your old one. This is true exactly when `score ≥ 0` *and* `bonus_pct > 0` — both conditions must hold.

### The dramatised bug

To show what `seal()` does when an assumption is missing, we deliberately rebuild the axiom set with the `score_nonneg` axiom dropped. The `Symbol("score", nonnegative=True)` constructor still carries the assumption silently, but the `AxiomSet` no longer declares it. Watch:


In [7]:
# Rebuild the axiom set MINUS score_nonneg (the simulated bug)
with unevaluated():
    broken_axioms = AxiomSet(
        name="nqg_neurons_broken",
        axioms=tuple(
            a for a in nqg_axioms.axioms if a.name != "score_nonneg"
        ),
    )

broken_h = broken_axioms.hypothesis(
    "trust_bonus_monotone",
    expr=sympy.Ge(score_new, score),
)
broken_script = (
    ProofBuilder(broken_axioms, broken_h.name,
                 name="trust_mono_broken",
                 claim="(broken) score' >= score with axiom set MISSING score_nonneg")
    .lemma(
        # EQUALITY using `score` — load-bearing audit flags by presence,
        # not by re-verification.  No assumptions dict here on purpose.
        "rewrite_factored",
        LemmaKind.EQUALITY,
        expr=score + (score / 100) * bonus_pct,
        expected=score * (1 + bonus_pct / 100),
        description="score + (score/100)*bonus_pct == score*(1 + bonus_pct/100)",
    )
    .lemma(
        "diff_nonneg",
        LemmaKind.QUERY,
        expr=sympy.Q.nonnegative((score / 100) * bonus_pct),
        assumptions={"score": {"nonnegative": True}, "bonus_pct": {"positive": True}},
        depends_on=["rewrite_factored"],
    )
    .build()
)

try:
    seal(broken_axioms, broken_h, broken_script)
    raise AssertionError("expected seal() to refuse, but it succeeded")
except ValueError as e:
    print(f"seal() refused:\n\n{e}")
    assert "load-bearing" in str(e), "expected load-bearing diagnostic"


seal() refused:

Load-bearing symbol assumptions found without matching axioms. These are hidden axioms — the proof depends on them but they are not declared in the axiom set:
  - Symbol 'score' has assumption 'nonnegative=True' which is load-bearing (affects lemmas ['rewrite_factored']) but is not declared as an axiom. Add an axiom for 'score' to the axiom set.


### Why this is the killer feature

The `nonnegative=True` flag on `score`'s constructor was about to be used by SymPy during lemma verification — but it was **not** declared as an axiom in the set hashed into the bundle. If `seal()` had let it through, the bundle would have certified the claim under an assumption that was nowhere on the receipt.

This is hidden assumption hunting in real time. The fix is to name the assumption: keep `Axiom(name="score_nonneg", expr=score >= 0)` in the axiom set. We already did this in our canonical `nqg_axioms`. Sealing REQ-TRUST-MONO against the proper axiom set succeeds.

The deeper point: in Rust, `score` was constructed by `min_max_normalize_result`, which by *construction* puts the value in `[0, 1]`. That's a proof obligation on `min_max_normalize_result`, and the moment we cite it in another proof we are inheriting it as an axiom — so it has to be declared. `seal()` makes that obligation visible.


In [8]:
mono_script = (
    ProofBuilder(
        nqg_axioms, h_mono.name,
        name="trust_mono_proof",
        claim="score' >= score via factor (1 + bonus_pct/100)",
    )
    .lemma(
        "rewrite_factored",
        LemmaKind.EQUALITY,
        expr=score + (score / 100) * bonus_pct,
        expected=score * (1 + bonus_pct / 100),
        description="score + (score/100)*bonus_pct == score*(1 + bonus_pct/100)",
    )
    .lemma(
        "diff_nonneg",
        LemmaKind.QUERY,
        expr=sympy.Q.nonnegative((score / 100) * bonus_pct),
        assumptions={"score": {"nonnegative": True}, "bonus_pct": {"positive": True}},
        depends_on=["rewrite_factored"],
        description="(score/100)*bonus_pct >= 0 if score>=0 and bonus_pct>0",
    )
    .build()
)
mono_bundle = seal(nqg_axioms, h_mono, mono_script)

print(f"REQ-TRUST-MONO")
print(f"  hash:   {mono_bundle.bundle_hash}")
print(f"  status: {mono_bundle.proof_result.status.value}")


REQ-TRUST-MONO
  hash:   b5dadb73f9c24a0bfea656a02b94d9a08df332f998deac85e75d14265a690d56
  status: VERIFIED


## Failure-and-fix #2 — SymPy's bounded-interval blind spot

REQ-TELEPORT-POS: prove that PageRank's "teleport" term is strictly positive.

Target lines in the Rust:

```rust
// neurons/src/trust_graph.rs::calculate_page_rank
let mut rank = (1.0 - damping_factor) / nodes.len() as f64;   // <-- the teleport
```

PageRank rebases each iteration with a uniform `(1−d)/N` term so every node has a positive baseline rank — that's how the random walk stays ergodic. The claim `(1−d)/N > 0` requires `d < 1` (axiom N5) and `N ≥ 1` (axiom N6).

### Plain-English math

If `d < 1`, then `1 − d > 0`. Dividing a positive number by a positive number gives a positive number. So `(1−d)/N > 0`. Three lines.

The natural first attempt is to write that chain as one `BOOLEAN` lemma with the implication baked in:

```python
Implies(
    And(d > 0, d < 1, N > 0),
    (1 - d) / N > 0,
)
```

Logically valid. But SymPy's `simplify` and `refine` together don't reliably chain a *strict* inequality (`d < 1`) through subtraction (`1 − d > 0`) and then through division by another positive symbol. This is a documented blind spot — strict inequalities and bounded intervals are where the Q-system gives up. Watch:


In [9]:
naive_teleport_script = (
    ProofBuilder(
        nqg_axioms, h_teleport.name,
        name="teleport_naive",
        claim="(1-d)/N > 0 via single BOOLEAN implication (will fail)",
    )
    .lemma(
        "teleport_implication",
        LemmaKind.BOOLEAN,
        expr=sympy.Implies(
            sympy.And(d > 0, d < 1, N > 0),
            (1 - d) / N > 0,
        ),
        description="single-shot BOOLEAN attempt",
    )
    .build()
)

try:
    seal(nqg_axioms, h_teleport, naive_teleport_script)
    raise AssertionError("expected seal() to refuse, but it succeeded")
except ValueError as e:
    msg = str(e)
    print(f"seal() refused:\n\n{msg}")
    assert "FAILED" in msg or "failed" in msg, "expected verify failure"


seal() refused:

verify_proof returned FAILED, not VERIFIED. Cannot seal an unverified proof. Summary: Lemma 'teleport_implication' failed: None


### The helper-symbol pattern, again

The trick used for the conviction-voting `REQ-PASS` proof works here too. Introduce a positive helper symbol `e := 1 − d`. Axiom N5 (`d < 1`) is exactly what makes `e > 0`. The teleport rewrites cleanly:

$$\frac{1 - d}{N} = \frac{e}{N}$$

Both `e` and `N` are positive symbols. The Q-system handles `Q.positive(e/N)` directly — no implication chaining needed.

**Notice the trade.** The naive `BOOLEAN(Implies(...))` form failed silently because SymPy couldn't verify the implication. The helper-symbol form succeeds because `e > 0` *is* axiom N5, made symbolic. The proof only seals when N5 is in the axiom set. Drop N5 and `e`'s positivity has nowhere to come from — and the load-bearing check would fire instead.


In [10]:
e_sym = Symbol("e", positive=True)   # e := 1 - d  (positive iff d < 1, i.e., axiom N5)
teleport_factored = e_sym / N

teleport_script = (
    ProofBuilder(
        nqg_axioms, h_teleport.name,
        name="teleport_positive_proof",
        claim="(1-d)/N > 0 via helper e = 1 - d positive (uses N5)",
    )
    .lemma(
        "rewrite_with_helper",
        LemmaKind.EQUALITY,
        expr=(1 - d) / N,
        expected=teleport_factored.subs(e_sym, 1 - d),
        description="(1-d)/N == e/N with e := 1-d",
    )
    .lemma(
        "factored_form_positive",
        LemmaKind.QUERY,
        expr=sympy.Q.positive(teleport_factored),
        assumptions={"e": {"positive": True}, "N": {"positive": True}},
        depends_on=["rewrite_with_helper"],
        description="e/N > 0 when e>0 and N>0; e>0 *is* axiom N5",
    )
    .build()
)
teleport_bundle = seal(nqg_axioms, h_teleport, teleport_script)

print(f"REQ-TELEPORT-POS")
print(f"  hash:   {teleport_bundle.bundle_hash}")
print(f"  status: {teleport_bundle.proof_result.status.value}")


REQ-TELEPORT-POS
  hash:   f80b6344d1277b59378dbbae8f12e7c8213fa64cb7abd8f5be89285743340ce2
  status: VERIFIED


In [11]:
# Plug in damping values that VIOLATE N5 (d < 1) to see the deployment
# regime the symbolic bundle does NOT certify.
print("Deployment regime sweep — what happens when N5 (d < 1) is violated?")
print()
print(f"{'d':<8}{'1-d':<10}{'(1-d)/N':<12}{'sign':<8}{'verdict'}")
print("-" * 60)
for damp in [Rational(85, 100), Rational(95, 100), Rational(99, 100),
             Rational(1, 1), Rational(105, 100), Rational(11, 10)]:
    one_minus_d = 1 - damp
    teleport_value = one_minus_d / 5  # 5-node graph
    if teleport_value > 0:
        sign, verdict = ">  0", "ergodic ok"
    elif teleport_value == 0:
        sign, verdict = "== 0", "BREAKS — no baseline"
    else:
        sign, verdict = "<  0", "BREAKS — negative rank"
    print(f"{str(damp):<8}{str(one_minus_d):<10}{str(teleport_value):<12}{sign:<8}{verdict}")

print()
print("d >= 1 collapses or inverts the teleport, which:")
print("  - removes the per-iteration baseline that makes every node reachable,")
print("  - violates the ergodicity premise of Perron-Frobenius,")
print("  - leaves the PageRank stationary distribution undefined.")
print()
print("Axiom N5 buys all three guarantees; only the first is proved here.")
print("The other two require a foundation bundle that imports Perron-Frobenius.")


Deployment regime sweep — what happens when N5 (d < 1) is violated?

d       1-d       (1-d)/N     sign    verdict
------------------------------------------------------------
17/20   3/20      3/100       >  0    ergodic ok
19/20   1/20      1/100       >  0    ergodic ok
99/100  1/100     1/500       >  0    ergodic ok
1       0         0           == 0    BREAKS — no baseline
21/20   -1/20     -1/100      <  0    BREAKS — negative rank
11/10   -1/10     -1/50       <  0    BREAKS — negative rank

d >= 1 collapses or inverts the teleport, which:
  - removes the per-iteration baseline that makes every node reachable,
  - violates the ergodicity premise of Perron-Frobenius,
  - leaves the PageRank stationary distribution undefined.

Axiom N5 buys all three guarantees; only the first is proved here.
The other two require a foundation bundle that imports Perron-Frobenius.


In [12]:
from symproof.export import proof_dag_mermaid

print("=" * 72)
print("  NQG  --  Requirements Traceability Matrix")
print("=" * 72)
print(f"  AxiomSet hash: {nqg_axioms.axiom_set_hash}")
print()

rtm = [
    ("REQ-RATIO-FLOOR",  "active_votes_ratio >= 0.5",        ratio_bundle),
    ("REQ-TRUST-MONO",   "trust-graph bonus is monotone",    mono_bundle),
    ("REQ-TELEPORT-POS", "PageRank teleport (1-d)/N > 0",    teleport_bundle),
]
for req, claim, b in rtm:
    print(f"  {req:<18s} {claim}")
    print(f"    hash:   {b.bundle_hash}")
    print(f"    lemmas: {len(b.proof.lemmas)}")
    print()

print("Framed but not proved here (need a Perron-Frobenius foundation):")
for req, name in [
    ("REQ-PR-IRREDUCIBLE", "PageRank transition matrix is irreducible under N5"),
    ("REQ-PR-UNIQUE",      "Stationary distribution is unique"),
    ("REQ-PR-CONVERGES",   "Power iteration converges to it"),
]:
    print(f"  {req:<22s} {name}")

print()
print("Proof DAG (paste into a Mermaid renderer to see):")
print()
print("```mermaid")
print(proof_dag_mermaid(ratio_bundle, mono_bundle, teleport_bundle))
print("```")


  NQG  --  Requirements Traceability Matrix
  AxiomSet hash: d1e5e56ad03d08ffdf04a318da3a4605aac5515e8b973420b99a1cf1c5f15a40

  REQ-RATIO-FLOOR    active_votes_ratio >= 0.5
    hash:   6143139b484d35431f96b732c0a777e68a2ca8b0e3141a4d07f2b69e3ec6d57a
    lemmas: 1

  REQ-TRUST-MONO     trust-graph bonus is monotone
    hash:   b5dadb73f9c24a0bfea656a02b94d9a08df332f998deac85e75d14265a690d56
    lemmas: 2

  REQ-TELEPORT-POS   PageRank teleport (1-d)/N > 0
    hash:   f80b6344d1277b59378dbbae8f12e7c8213fa64cb7abd8f5be89285743340ce2
    lemmas: 2

Framed but not proved here (need a Perron-Frobenius foundation):
  REQ-PR-IRREDUCIBLE     PageRank transition matrix is irreducible under N5
  REQ-PR-UNIQUE          Stationary distribution is unique
  REQ-PR-CONVERGES       Power iteration converges to it

Proof DAG (paste into a Mermaid renderer to see):

```mermaid
graph BT
    6143139b484d3543["max_ge_first_1/2_ratio<br/>max_1/2_ratio_ge_1/2"]
    style 6143139b484d3543 fill:#d4edda,stroke:

## Where to go from here

Three follow-on directions for an NQG audit, each strengthening a different leg of the V&V program described in the symproof [README](../README.md#the-three-layer-architecture).

### (a) Software requirements for code audit

Each axiom and each sealed hypothesis becomes a row in a requirements traceability matrix that auditors can use against the Rust and Soroban code:

- **Input validation** from the axioms. The off-chain WASM entrypoint should reject inputs where damping `d` is outside `(0, 1)`, where `total_votes` is zero (would produce NaN ratios), or where `nodes.len() == 0`. Each becomes a `bail!()` with a comment pointing at the bundle hash here.
- **State invariants** from the hypotheses. `min_max_normalize_result` produces values in `[0, 1]` is something the Rust unit tests should assert on every output, and a `proptest` should fuzz against. The hash of REQ-TRUST-MONO is the spec the test is *for*.
- **Lemma-level postconditions** from the proof chain. Each `EQUALITY` lemma is an algebraic claim a Rust function should preserve at runtime — these are the cheapest property tests to write.

### (b) Foundations and the hidden axiom budget

The "framed" rows in the catalog (REQ-PR-IRREDUCIBLE, REQ-PR-UNIQUE, REQ-PR-CONVERGES) are where this audit hits the wall. Two paths forward:

- **Cite Perron–Frobenius as an axiom** (`expr=sympy.S.true` plus a `Citation`). Cheap, but every downstream proof inherits all three results without surfacing their conditions.
- **Build a `library.markov` foundation** that proves the algebraic kernel of Perron–Frobenius for column-stochastic matrices, and uses `seal(foundations=...)` to enforce that downstream PageRank proofs declare every condition (column-stochasticity of the trust matrix, primitivity, damping). This is the [DIP-routing pattern](../symproof/library/examples/dip_routing/README.md) — a one-time investment that makes every PageRank-based audit henceforth more honest.

### (c) `lambdify` for parameter calibration and dashboards

The axiom set and proven claims are SymPy expressions. `sympy.lambdify` turns each into a NumPy callable. Three uses:

- **Parameter regime visualization.** Sweep `(d, N)` and shade the region where N5 holds. Useful for dashboards that operations needs to see when proposing parameter changes.
- **Calibration tool.** Given operator-level targets (e.g. "at least 5% baseline per node, max 100-node graph"), solve for `d` that satisfies every axiom plus the targets.
- **Cross-check against simulation.** The symbolic proof tells the simulator which invariants to assert; the simulator tells the prover when the framing has missed something.

These three legs — code, foundations, dashboards — turn a notebook into an audit deliverable.


## References

- Stellar Community Fund contracts: <https://github.com/stellar/stellar-community-fund-contracts>
- Neural Quorum Governance handbook: <https://stellarcommunityfund.gitbook.io/scf-handbook/community-involvement/governance/neural-quorum-governance>
- Sister walkthrough this notebook clones: [`examples/conviction_voting.ipynb`](conviction_voting.ipynb)
- Canonical script form: [`examples/nqg_voting.py`](nqg_voting.py)
- Hidden-axiom worked example with foundations: [`symproof/library/examples/dip_routing/`](../symproof/library/examples/dip_routing/)
- V&V three-layer architecture: see the project [README](../README.md#the-three-layer-architecture)
- PageRank original paper: Page, Brin, Motwani, Winograd, "The PageRank Citation Ranking" (1999)
